# Trabalho Prático 1 - Aprendizagem Automática
## Previsão de Preços de Carros Usados (Kaggle Competition)

**Licenciatura em Engenharia de Sistemas e Tecnologias Informáticas** 
**Unidade Curricular:** Aprendizagem Automática  
**Ano Letivo:** 2025/2026 

---

### Introdução e Objetivos
Este *notebook* documenta o processo de desenvolvimento de modelos de Aprendizagem Automática para prever o preço de carros usados, no âmbito da competição Kaggle da disciplina.

O presente trabalho tem como objetivo o desenvolvimento de um modelo de regressão capaz de prever o preço de automóveis usados com base num conjunto de características intrínsecas (ex: marca, motorização, quilometragem). A metodologia adotada foca-se no **pré-processamento avançado de dados** e na utilização de **técnicas de Ensemble Learning**.

A abordagem apresentada nesta versão corrige problemas estruturais de versões anteriores, especificamente a fuga de dados (*data leakage*) e erros de imputação (`Input X contains NaN`), através da implementação de **Pipelines do Scikit-Learn**.

### Metodologia Aplicada:
1.  **Engenharia de Atributos:** Extração rigorosa de dados técnicos da coluna `engine` e normalização de categorias (cores e transmissão), baseada na análise exploratória prévia.
2.  **Imputação Hierárquica:** Tratamento de valores omissos utilizando a média/moda agrupada por marca e modelo, preservando a coerência do domínio automóvel.
3.  **Modelagem Híbrida:** Utilização de um *Ensemble* ponderado constituído por XGBoost, Random Forest e Gradient Boosting.
4.  **Pipeline Robusto:** Integração de `Imputer`, `Scaler` e `Model` num fluxo contínuo para garantir a integridade dos dados em validação cruzada.

## 1. Configuração do Ambiente e Importação de Bibliotecas

Nesta etapa, importam-se as bibliotecas fundamentais para a manipulação de dados (`pandas`, `numpy`) e para a construção dos modelos (`sklearn`, `xgboost`). 

Destaca-se a definição de uma semente aleatória (`RANDOM_STATE = 42`) para garantir a **reprodutibilidade** dos resultados experimentais, um requisito essencial em trabalhos académicos. O modo de execução pode ser alternado entre `quick` (para testes rápidos) e `full` (para treino intensivo).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os, joblib, re, random, datetime

from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

# --- CONFIGURAÇÃO GLOBAL ---
MODE = 'full'  # Opções: 'quick' (testes) ou 'full' (produção)
TRAIN_N_JOBS = -1
RANDOM_STATE = 42

# Definição de hiperparâmetros baseados no modo de execução
if MODE == 'quick':
    CV_FOLDS = 2
    N_ITER = 2
    XGB_ESTIMATORS = 50
else:
    CV_FOLDS = 5
    N_ITER = 15
    XGB_ESTIMATORS = 3000

# Garantia de Reprodutibilidade
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

print(f"Ambiente Configurado. Modo: {MODE} | Folds: {CV_FOLDS}")

Ambiente Configurado. Modo: full | Folds: 5


## 2. Carregamento e Pré-visualização dos Dados

Procede-se à leitura dos conjuntos de dados de treino e teste. É fundamental verificar a integridade destes ficheiros antes de iniciar o processamento.

In [7]:
try:
    train_df = pd.read_csv('../data/train.csv')
    test_df = pd.read_csv('../data/test.csv')
    print(f"Sucesso: Dados carregados da pasta 'data'.")
    print(f"Dimensões: Treino {train_df.shape}, Teste {test_df.shape}")
except FileNotFoundError:
    print("Erro Crítico: Ficheiros não encontrados na pasta 'data'.")
    print("Confirme se a estrutura é: 'seu_notebook.ipynb' e uma pasta 'data' ao lado contendo os csv.")

Sucesso: Dados carregados da pasta 'data'.
Dimensões: Treino (188533, 13), Teste (125690, 12)


## 3. Engenharia de Atributos e Limpeza de Dados

Esta secção contém a lógica central de tratamento de dados, transposta da análise exploratória prévia. As funções implementadas visam estruturar dados não estruturados e reduzir a cardinalidade de variáveis categóricas.

### 3.1. Extração de Características do Motor (`clean_engine_data`)
A coluna `engine` contém strings complexas (ex: "4.0L V8 32V GDI DOHC Twin Turbo"). Utilizamos **Expressões Regulares (Regex)** para extrair três variáveis numéricas cruciais:
* **HP (Cavalos):** Potência do veículo.
* **Liters:** Cilindrada do motor.
* **Cylinders:** Número de cilindros.

### 3.2. Normalização de Categorias
Para evitar o problema da "maldição da dimensionalidade", aplicamos dicionários de mapeamento:
* **Transmissão:** Agrupada em 'Automatic', 'Manual', 'CVT' e 'Unknown'.
* **Cores (Exterior e Interior):** Agrupadas em famílias (ex: 'Jet Black', 'Ebony' $\rightarrow$ 'Black'; 'Metallic' mantém-se como categoria de acabamento).
* **Acidentes e Títulos:** Conversão para binário (0/1) para facilitar a interpretação pelos modelos.

In [8]:
# --- DICIONÁRIOS DE MAPEAMENTO (Baseado na Análise Exploratória) ---

transmission_map = {
    'automatic': ['1-Speed A/T', '10-Speed A/T', '10-Speed Automatic', '2-Speed A/T', '4-Speed Automatic', 
                  '5-Speed Automatic', '6-Speed A/T', '6-Speed Automatic', '7-Speed A/T', '7-Speed Automatic', 
                  '8-Speed A/T', '8-Speed Automatic', '9-Speed Automatic', 'A/T', 'Automatic', 'Automatic CVT'],
    'manual': ['5-Speed M/T', '6-Speed M/T', '6-Speed Manual', '7-Speed Manual', 'Manual', 'M/T'],
    'cvt': ['CVT Transmission', 'CVT-F', 'Variable'],
    'unknown': ['–', 'F', 'SCHEDULED FOR OR IN PRODUCTION', '2']
}

ext_col_map = {
    'Basic': ['black', 'white', 'gray', 'blue', 'red', 'green', 'yellow', 'brown', 'orange', 'beige', 'pink', 'purple'],
    'Metallic': ['metallic'],
    'Matte': ['matte'],
    'Pearl': ['pearl', 'pearlcoat'],
    'Premium' : ['tintcoat', 'tri-coat', 'clearcoat', 'effect', 'mica', 'bright'],
    'Special': []
}

color_keywords = {
    'Red': ['red', 'crimson', 'ruby', 'hotspur', 'garnet', 'rosso'],
    'Blue': ['blue', 'blu', 'navy', 'cypress', 'azure', 'sky'],
    'Green': ['green', 'emerald', 'forest', 'olive'],
    'Brown': ['brown', 'chocolate', 'walnut', 'chestnut', 'brandy', 'roast'],
    'Beige': ['beige', 'parchment', 'camel', 'tan', 'macchiato', 'shale'],
    'Gray': ['gray', 'grey', 'slate', 'silver', 'gunmetal', 'ash', 'cloud', 'stone'],
    'White': ['white', 'ivory', 'pearl', 'snow', 'alabaster', 'oyster', 'linen', 'ice'],
    'Black': ['nero', 'black', 'jet', 'graphite', 'ebony', 'onyx', 'charcoal', 'obsidian', 'dark'],
    'Metallic': ['metallic', 'platinum', 'caviar', 'stormy']
}

# --- FUNÇÕES AUXILIARES ---

def clean_engine_data(row):
    """Extrai HP, Litros e Cilindros da string do motor usando Regex."""
    engine = str(row['engine'])
    hp, liters, cylinders = 0, 0, 0
    
    # Extração de HP
    hp_match = re.search(r'(\d+\.?\d*)HP', engine)
    if hp_match: hp = float(hp_match.group(1))
    
    # Extração de Litros
    liters_match = re.search(r'(\d+\.?\d*)\s*Liter', engine)
    if not liters_match: liters_match = re.search(r'(\d+\.?\d*)L', engine)
    if liters_match: liters = float(liters_match.group(1))
    
    # Extração de Cilindros
    cylinders_match = re.search(r'(\d+) Cylinder', engine)
    if not cylinders_match: cylinders_match = re.search(r'V(\d)', engine)
    if cylinders_match: cylinders = float(cylinders_match.group(1))
    
    # Correção para elétricos (se detetado)
    if 'Electric' in engine and liters == 0 and cylinders == 0:
        liters, cylinders = 0, 0

    return pd.Series([hp, liters, cylinders])

def map_transmission_category(transmission):
    """Normaliza a transmissão."""
    transmission = str(transmission)
    for category, terms in transmission_map.items():
        if transmission in terms:
            return category
    return 'unknown'

def map_ext_col_category(ext_col):
    """Normaliza a cor exterior."""
    ext_col = str(ext_col).lower()
    if ext_col in ['c / c', '–']: return 'Unknown'
    
    # Verifica categorias especiais primeiro
    for category, keywords in ext_col_map.items():
        if category != 'Basic' and any(k in ext_col for k in keywords):
            return category
    
    # Verifica cores básicas
    for basic_color in ext_col_map['Basic']:
        if basic_color in ext_col:
            return 'Basic'
            
    return 'Other'

def categorize_int_color(color_name):
    """Normaliza a cor interior."""
    color_name = str(color_name).lower().strip()
    for color, keywords in color_keywords.items():
        if any(keyword in color_name for keyword in keywords):
            return color
    return 'Other'

def process_dataframe(df):
    """Função mestre que aplica todas as transformações ao DataFrame."""
    data = df.copy()
    
    # 1. Extração Motor
    data[['HP', 'Liters', 'Cylinders']] = data.apply(clean_engine_data, axis=1)
    
    # 2. Tratamento de Strings e Limpeza Básica
    data['brand'] = data['brand'].astype(str)
    data['model'] = data['model'].astype(str)
    
    # 3. Transmissão
    data['transmission'] = data['transmission'].apply(map_transmission_category)
    
    # 4. Cores
    data['ext_col'] = data['ext_col'].apply(map_ext_col_category)
    data['int_col'] = data['int_col'].apply(categorize_int_color)
    
    # 5. Acidentes e Título (Binário)
    data['accident'] = data['accident'].apply(lambda x: 1 if 'At least 1' in str(x) else 0)
    data['clean_title'] = data['clean_title'].apply(lambda x: 1 if str(x).lower() == 'yes' else 0)
    
    # 6. Combustível (Simplificação)
    data['fuel_type'] = data['fuel_type'].apply(lambda x: 'Electric' if 'Electric' in str(x) 
                                                else ('Hybrid' if 'Hybrid' in str(x) 
                                                else ('Diesel' if 'Diesel' in str(x) else 'Gasoline')))

    # 7. Idade do Carro
    data['model_year'] = pd.to_numeric(data['model_year'], errors='coerce')
    data['car_age'] = datetime.datetime.now().year - data['model_year']
    
    # Tratamento de zeros gerados pelo regex (convertemos em NaN para o Imputer tratar depois)
    cols_to_fix = ['HP', 'Liters', 'Cylinders']
    for col in cols_to_fix:
        data[col] = data[col].replace(0, np.nan)
        
    return data

# Aplicar Processamento
print("A aplicar engenharia de atributos...")
train_processed = process_dataframe(train_df)
test_processed = process_dataframe(test_df)
print("Engenharia de atributos concluída.")

A aplicar engenharia de atributos...
Engenharia de atributos concluída.


## 4. Imputação Inteligente de Dados

A presença de valores nulos (NaN) é tratada utilizando conhecimento de domínio. Em vez de utilizar uma média global simples, opta-se por uma **imputação hierárquica por marca**. 

Por exemplo, se o valor de `HP` (cavalos) estiver em falta num *Porsche*, será preenchido com a média de HP de outros *Porsches*, e não com a média global de todos os carros, o que preserva as características da marca.

In [9]:
# --- LÓGICA DE IMPUTAÇÃO POR MARCA ---

# Calcular médias por marca no conjunto de treino (para evitar Data Leakage)
brand_hp_mean = train_processed.groupby('brand')['HP'].mean()
global_hp_mean = train_processed['HP'].mean()

brand_liters_mean = train_processed.groupby('brand')['Liters'].mean()
global_liters_mean = train_processed['Liters'].mean()

def fill_missing_values(row, col_name, brand_map, global_val):
    """Preenche valores nulos com a média da marca; se não existir, usa a média global."""
    if pd.isna(row[col_name]):
        return brand_map.get(row['brand'], global_val)
    return row[col_name]

# Aplicar Imputação
for df in [train_processed, test_processed]:
    # HP e Litros via Marca
    df['HP'] = df.apply(lambda x: fill_missing_values(x, 'HP', brand_hp_mean, global_hp_mean), axis=1)
    df['Liters'] = df.apply(lambda x: fill_missing_values(x, 'Liters', brand_liters_mean, global_liters_mean), axis=1)
    
    # Cilindros: preenchimento simples com mediana (discreto)
    df['Cylinders'] = df['Cylinders'].fillna(train_processed['Cylinders'].median())
    
    # Engenharia Adicional pós-imputação
    df['hp_per_liter'] = df['HP'] / df['Liters'].replace(0, 1) # Evitar divisão por zero

print("Imputação de dados técnicos concluída.")

Imputação de dados técnicos concluída.


## 5. Codificação de Variáveis Categóricas (Encoding)

Os algoritmos de Machine Learning requerem entradas numéricas. Utilizamos o `LabelEncoder` para transformar variáveis categóricas em números inteiros. 

**Nota Importante:** Para garantir consistência entre o treino e o teste, o `LabelEncoder` é ajustado (`fit`) na união de todos os valores categóricos presentes em ambos os conjuntos de dados.

In [10]:
label_cols = ['brand', 'model', 'fuel_type', 'transmission', 'ext_col', 'int_col']

label_encoders = {}

for col in label_cols:
    le = LabelEncoder()
    # Combinar treino e teste para garantir que todas as categorias são conhecidas
    # Convertemos para string para evitar erros de tipos mistos
    all_values = pd.concat([train_processed[col].astype(str), test_processed[col].astype(str)]).unique()
    le.fit(all_values)
    
    train_processed[col] = le.transform(train_processed[col].astype(str))
    test_processed[col] = le.transform(test_processed[col].astype(str))
    
    label_encoders[col] = le

print("Label Encoding aplicado com sucesso.")

Label Encoding aplicado com sucesso.


## 6. Preparação para Modelação e Pipelines

Nesta fase, separam-se as variáveis independentes ($X$) da variável alvo ($y$). 

**Transformação do Alvo:** Aplica-se uma transformação logarítmica (`np.log1p`) à variável `price`. Isto é crucial em regressões financeiras porque os preços tendem a ter uma distribuição assimétrica (cauda longa à direita). O logaritmo normaliza a distribuição, melhorando a performance dos modelos lineares e baseados em árvores.

**Pipeline de Scikit-Learn:** Definimos um `ColumnTransformer` que aplica `SimpleImputer` e `RobustScaler`. O `RobustScaler` é preferido aqui por ser menos sensível a *outliers* do que o `StandardScaler`.

In [11]:
# Seleção de Features Finais
features = ['brand', 'model', 'car_age', 'milage', 'HP', 'Liters', 'Cylinders', 'hp_per_liter',
            'fuel_type', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title']

X = train_processed[features]
y = np.log1p(train_processed['price']) # Transformação Logarítmica

X_test_final = test_processed[features]

# Divisão Treino/Validação (Hold-out)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

# Definição do Pré-processador (Pipeline)
# O pipeline garante que a imputação e o scaling ocorrem dentro de cada fold da validação cruzada
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')), # Segurança extra contra NaNs
            ('scaler', RobustScaler()) # RobustScaler lida melhor com outliers
        ]), features)
    ])

print("Dados preparados e Pipeline de pré-processamento definido.")

Dados preparados e Pipeline de pré-processamento definido.


## 7. Treino de Modelos e Otimização de Hiperparâmetros

Utiliza-se uma abordagem de **Ensemble** combinando três algoritmos robustos:
1.  **XGBoost:** Algoritmo de *Gradient Boosting* altamente eficiente.
2.  **Random Forest:** Método de *Bagging* que reduz a variância.
3.  **Gradient Boosting Regressor (sklearn):** Complemento ao XGBoost.

Para cada modelo, utiliza-se `RandomizedSearchCV` para encontrar a melhor combinação de hiperparâmetros. O uso de `Pipeline` aqui é crítico para evitar o erro `Input X contains NaN`, assegurando que os dados passam pelo `preprocessor` antes de chegarem ao modelo.

In [ ]:
models = []

# 1. XGBoost
xgb_pipe = Pipeline([('pre', preprocessor), ('model', XGBRegressor(objective='reg:squarederror', n_jobs=TRAIN_N_JOBS, random_state=RANDOM_STATE))])
xgb_params = {
    'model__n_estimators': [XGB_ESTIMATORS],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [6, 8, 10],
    'model__subsample': [0.7, 0.8],
    'model__colsample_bytree': [0.7, 0.8],
    'model__reg_alpha': [0.1, 0.5],
    'model__reg_lambda': [1.0, 1.5]
}
models.append(('XGB', xgb_pipe, xgb_params))

# 2. Random Forest
rf_pipe = Pipeline([('pre', preprocessor), ('model', RandomForestRegressor(n_jobs=TRAIN_N_JOBS, random_state=RANDOM_STATE))])
rf_params = {
    'model__n_estimators': [300, 500],
    'model__max_depth': [15, 20, None],
    'model__min_samples_leaf': [2, 4]
}
models.append(('RF', rf_pipe, rf_params))

# 3. Gradient Boosting
gb_pipe = Pipeline([('pre', preprocessor), ('model', GradientBoostingRegressor(random_state=RANDOM_STATE))])
gb_params = {
    'model__n_estimators': [500],
    'model__learning_rate': [0.05],
    'model__max_depth': [5],
    'model__subsample': [0.8]
}
models.append(('GBR', gb_pipe, gb_params))

# --- LOOP DE TREINO ---
trained_estimators = []
cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print("A iniciar processo de treino e otimização...\n")

for name, pipeline, params in models:
    print(f">>> A treinar {name}...")
    search = RandomizedSearchCV(pipeline, params, n_iter=N_ITER, cv=cv, 
                                scoring='neg_mean_squared_error', n_jobs=TRAIN_N_JOBS, 
                                random_state=RANDOM_STATE, verbose=1)
    
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    
    # Avaliação no conjunto de validação
    preds = best_model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(preds)))
    print(f"   -> RMSE Validação: {rmse:,.2f}")
    
    trained_estimators.append((name, best_model))
    
    # Guardar modelo
    joblib.dump(best_model, f'best_model_{name}.pkl')

A iniciar processo de treino e otimização...

>>> A treinar XGB...
Fitting 5 folds for each of 15 candidates, totalling 75 fits


## 8. Ensemble Final e Submissão

Para aumentar a robustez das previsões e reduzir o sobreajuste (*overfitting*), combinam-se as previsões dos três modelos. Foi atribuído um peso maior ao **XGBoost (60%)**, dado o seu desempenho historicamente superior em dados tabulares, seguido do Random Forest (30%) e Gradient Boosting (10%).

Os preços finais são revertidos da escala logarítmica usando `np.expm1` e guardados no ficheiro de submissão.

In [ ]:
print("\nCalculando Ensemble Ponderado...")

# Pesos definidos empiricamente com base na performance individual
weights = {'XGB': 0.6, 'RF': 0.3, 'GBR': 0.1}

final_test_predictions = np.zeros(len(X_test_final))
final_val_predictions = np.zeros(len(X_val))

for name, model in trained_estimators:
    weight = weights.get(name, 0)
    print(f" -> Adicionando contribuição de {name} (Peso: {weight})")
    
    # Prever Validação (para verificação do RMSE combinado)
    val_pred = model.predict(X_val)
    final_val_predictions += val_pred * weight
    
    # Prever Teste Final
    test_pred = model.predict(X_test_final)
    final_test_predictions += test_pred * weight

# Avaliação Final do Ensemble
rmse_ens = np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(final_val_predictions)))
print(f"\nRMSE DO ENSEMBLE (Validação): {rmse_ens:,.2f}")

# --- GERAÇÃO DO FICHEIRO DE SUBMISSÃO ---
final_price = np.expm1(final_test_predictions) # Reverter Log

submission = pd.DataFrame({
    'id': test_df['id'], 
    'price': np.clip(final_price, 0, None) # Garantir que não há preços negativos
})

submission.to_csv('submission_v9_ensemble.csv', index=False)
print("Ficheiro 'submission_v9_ensemble.csv' gerado com sucesso.")